# Lab 13.4: Why Did the Agent Go Around?

Chapter 13 of *Explainable AI in Practice*. A delivery van on a 5 x 7 grid must reach a depot. A congested city block lies on the direct road: there, 30 percent of moves fail (the van waits), fuel use is higher and each step carries a lateness-risk penalty. We solve the task by value iteration, explain the policy with decomposed rewards, summarize it, and test the explanation by changing the world. Numpy only.

In [1]:
import numpy as np
H, W, GOAL = 5, 7, (2, 6)
CITY = {(r, c) for r in (1, 2, 3) for c in (2, 3, 4)}
ACTS = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}
KINDS = ["time", "fuel", "lateness risk"]
S = [(r, c) for r in range(H) for c in range(W)]

def step(s, a, risk=1.2, slip=0.3):
    if s == GOAL:
        return [(s, 1.0, np.zeros(3))]
    dr, dc = ACTS[a]
    nxt = (min(max(s[0] + dr, 0), H - 1), min(max(s[1] + dc, 0), W - 1))
    r = np.array([-1.0, -0.6 if nxt in CITY else -0.4, -risk if nxt in CITY else 0.0])
    return [(nxt, 1 - slip, r), (s, slip, r)] if s in CITY else [(nxt, 1.0, r)]

def solve(gamma=0.97, **kw):
    V = {s: 0.0 for s in S}
    for _ in range(500):
        V = {s: max(sum(p * (r.sum() + gamma * V[n]) for n, p, r in step(s, a, **kw)) for a in ACTS) for s in S}
    pi = {s: max(ACTS, key=lambda a: sum(p * (r.sum() + gamma * V[n]) for n, p, r in step(s, a, **kw))) for s in S}
    Vc = {s: np.zeros(3) for s in S}
    for _ in range(500):
        Vc = {s: sum(p * (r + gamma * Vc[n]) for n, p, r in step(s, pi[s], **kw)) for s in S}
    Q = lambda s, a: sum(p * (r + gamma * Vc[n]) for n, p, r in step(s, a, **kw))
    return pi, Q

pi, Q = solve()
def path(pi, s=(2, 1)):
    out = [s]
    while s != GOAL and len(out) < 20:
        dr, dc = ACTS[pi[s]]; s = (s[0] + dr, s[1] + dc); out.append(s)
    return out
print("Checkpoint 1: route from A (2, 1):", path(pi))

Checkpoint 1: route from A (2, 1): [(2, 1), (1, 1), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (1, 5), (2, 5), (2, 6)]


## 1. Contrastive explanations from decomposed Q-values

The minimal sufficient explanation (Juozapaitis et al., 2019) is the smallest set of reward kinds whose advantages for the chosen action outweigh all of its disadvantages.

In [2]:
def explain(s, alt):
    best = pi[s]
    diff = Q(s, best) - Q(s, alt)
    need = -diff[diff < 0].sum(); acc = 0.0; msx = []
    for i in np.argsort(-diff):
        if diff[i] <= 0: break
        msx.append(KINDS[i]); acc += diff[i]
        if acc > need: break
    print(f"state {s}: '{best}' over '{alt}':", {k: round(float(d), 2) for k, d in zip(KINDS, diff)},
          "total", round(float(diff.sum()), 2), "MSX+", msx)
print("Checkpoint 2:")
explain((2, 1), "right")
explain((0, 4), "down")
explain((1, 5), "left")

Checkpoint 2:
state (2, 1): 'up' over 'right': {'time': -2.21, 'fuel': -0.15, 'lateness risk': 4.41} total 2.06 MSX+ ['lateness risk']
state (0, 4): 'right' over 'down': {'time': 0.37, 'fuel': 0.35, 'lateness risk': 1.2} total 1.92 MSX+ ['lateness risk']
state (1, 5): 'down' over 'left': {'time': 2.23, 'fuel': 1.09, 'lateness risk': 1.2} total 4.52 MSX+ ['time']


## 2. A policy summary in the spirit of HIGHLIGHTS (Amir and Amir, 2018)

Importance of a state = best action value minus worst action value. The states where the choice matters most are the ones to show a person.

In [3]:
imp = {s: max(Q(s, a).sum() for a in ACTS) - min(Q(s, a).sum() for a in ACTS) for s in S if s != GOAL}
top = sorted(imp, key=imp.get, reverse=True)[:5]
print("Checkpoint 3: five most important states:")
for s in top:
    print(" ", s, "action", pi[s], "importance", round(imp[s], 2), "(in city block)" if s in CITY else "")

Checkpoint 3: five most important states:
  (2, 4) action right importance 5.29 (in city block)
  (1, 4) action right importance 5.15 (in city block)
  (3, 4) action right importance 5.15 (in city block)
  (2, 3) action right importance 5.06 (in city block)
  (2, 5) action right importance 4.62 


## 3. Test the explanation by changing the world

The explanation says the van goes around because of lateness risk. If that is true, removing the risk should send it straight through the block. If instead we only make the block free-flowing (no failed moves) but keep the risk, the explanation predicts it still goes around.

In [4]:
for label, kw in (("risk removed", dict(risk=0.0)), ("no congestion, risk kept", dict(slip=0.0)), ("both removed", dict(risk=0.0, slip=0.0))):
    p2, _ = solve(**kw)
    print(f"Checkpoint 4: {label:>26}: route {path(p2)}")

Checkpoint 4:               risk removed: route [(2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6)]


Checkpoint 4:   no congestion, risk kept: route [(2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6)]
Checkpoint 4:               both removed: route [(2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6)]


## Optional edit

Change `risk=1.2` in `step` to 0.6 and rerun everything. Which explanation changes first: the action, or the size of the lateness-risk advantage? Write two sentences on what that tells you about explanations near a decision boundary.